# 001b — Prepare Test Data (status_SSL)

測試資料來源：`status_SSL`（半監督學習標籤）

條件：
- `cough_detected > 0.8`
- 有 `status_SSL` 標籤
- **無**任何專家標註（`diagnosis_1~4` 全為 NaN）→ 與 Train 不重疊
- 無 `status_SSL` 或只有自報 `status` → 直接刪除

不做 balancing，保持真實分布。

## Imports

In [1]:
import os
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from utils import preproces

c:\Users\aint\anaconda3\envs\base1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 設定路徑

In [2]:
COUGHVID_ROOT = 'C:/Users/aint/Desktop/RNN/Final_project/coughvid_wav'
METADATA_CSV  = 'C:/Users/aint/Desktop/RNN/Final_project/metadata_compiled.csv'
COUGH_DETECTED_THRESHOLD = 0.8
OUTPUT_DIR = 'data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 讀取 metadata

In [3]:
df_meta = pd.read_csv(METADATA_CSV)
print('原始樣本數:', len(df_meta))

# 過濾低信心咳嗽聲
df_meta = df_meta[df_meta['cough_detected'] > COUGH_DETECTED_THRESHOLD].copy()
print('cough_detected 過濾後:', len(df_meta))

原始樣本數: 34434
cough_detected 過濾後: 18541


## 篩選 Test 樣本

條件：
1. 有 `status_SSL` 標籤
2. 無任何專家標註（避免與 Train 重疊）
3. 刪除只有自報 `status` 的樣本

In [4]:
EXPERT_COLS = ['diagnosis_1', 'diagnosis_2', 'diagnosis_3', 'diagnosis_4']
VALID_SSL   = ['COVID-19', 'symptomatic', 'healthy']

# 有 status_SSL
df_test = df_meta[df_meta['status_SSL'].isin(VALID_SSL)].copy()
print('有 status_SSL 樣本數:', len(df_test))

# 無任何專家標註（與 Train 不重疊）
has_expert = df_test[EXPERT_COLS].notna().any(axis=1)
df_test = df_test[~has_expert].copy()
print('排除有專家標註後:', len(df_test))

print('\nstatus_SSL 分佈（真實分布，不做 balancing）:')
print(df_test['status_SSL'].value_counts())

有 status_SSL 樣本數: 8331
排除有專家標註後: 7313

status_SSL 分佈（真實分布，不做 balancing）:
status_SSL
healthy     7263
COVID-19      50
Name: count, dtype: int64


## status_SSL → 三分類對應

In [5]:
SSL_LABEL_MAP = {
    'COVID-19':    'covid',
    'symptomatic': 'symptomatic',
    'healthy':     'healthy',
}
df_test['label'] = df_test['status_SSL'].map(SSL_LABEL_MAP)

print('label 分佈:')
print(df_test['label'].value_counts())

label 分佈:
label
healthy    7263
covid        50
Name: count, dtype: int64


## 確認 .wav 檔案存在

In [6]:
def find_wav(uuid):
    path = os.path.join(COUGHVID_ROOT, f'{uuid}.wav')
    return path if os.path.exists(path) else None

df_test['file_path'] = df_test['uuid'].apply(find_wav)
missing = df_test['file_path'].isna().sum()
print(f'找不到 .wav 的樣本數: {missing}（將被移除）')
df_test = df_test.dropna(subset=['file_path']).reset_index(drop=True)
print(f'最終可用樣本數: {len(df_test)}')

找不到 .wav 的樣本數: 318（將被移除）
最終可用樣本數: 6995


## 特徵提取

In [7]:
FEATURE_COLS = [
    'filename', 'chroma_stft', 'rmse', 'spectral_centroid', 'spectral_bandwidth',
    'rolloff', 'zero_crossing_rate',
    *[f'mfcc{i}' for i in range(1, 21)],
    'label'
]

feature_rows = []
errors = []

for _, row in tqdm(df_test.iterrows(), total=len(df_test)):
    try:
        feat = preproces(row['file_path'])
        if isinstance(feat, pd.Series):
            feat = feat.to_dict()
        feat['filename'] = os.path.basename(row['file_path'])
        feat['label']    = row['label']
        feature_rows.append(feat)
    except Exception as e:
        errors.append({'file': row['file_path'], 'error': str(e)})

print(f'成功: {len(feature_rows)} 筆，失敗: {len(errors)} 筆')

df_features = pd.DataFrame(feature_rows).reindex(columns=FEATURE_COLS)
out_path = os.path.join(OUTPUT_DIR, 'prepared_test_coughvid.csv')
df_features.to_csv(out_path, index=False)
print(f'已儲存至 {out_path}')
print(df_features['label'].value_counts())
df_features.head()

  5%|▌         | 350/6995 [00:13<03:57, 28.02it/s]c:\Users\aint\anaconda3\envs\base1\Lib\site-packages\librosa\core\pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
100%|██████████| 6995/6995 [04:05<00:00, 28.55it/s]


成功: 6995 筆，失敗: 0 筆
已儲存至 data\prepared_test_coughvid.csv
label
healthy    6956
covid        39
Name: count, dtype: int64


,filename,chroma_stft,rmse,spectral_centroid,spectral_bandwidth,rolloff,zero_crossing_rate,mfcc1,mfcc2,mfcc3,...,mfcc12,mfcc13,mfcc14,mfcc15,mfcc16,mfcc17,mfcc18,mfcc19,mfcc20,label
0,00039425-7f3a-42aa-ac13-834aaa2b6b92.wav,0.141164,0.018788,671.739573,602.743410,1442.026774,0.039734,-549.156738,24.094067,-6.318573,...,-4.920515,2.057861,0.897011,-1.837406,-3.557902,1.099849,-1.784741,-0.029227,0.182249,healthy
1,0009eb28-d8be-4dc1-92bb-907e53bc5c7a.wav,0.392828,0.097451,1684.497228,1611.580858,3496.537104,0.087128,-281.694855,104.776176,-29.471004,...,-5.137051,0.506576,-0.286182,-7.571201,-7.208370,-0.445022,-8.474636,-6.395027,-5.831004,healthy
2,001328dc-ea5d-4847-9ccf-c5aa2a3f2d0f.wav,0.513593,0.051830,2736.828237,1857.201647,4988.525391,0.277145,-399.854218,53.067383,-57.985279,...,-16.851395,0.961044,-3.153154,-0.951281,-6.794377,7.223249,-4.327006,-0.995339,-1.005051,healthy
3,0028b68c-aca4-4f4f-bb1d-cb4ed5bbd952.wav,0.285591,0.026696,1795.577539,1100.826360,3061.852010,0.112996,-487.515259,21.970707,-22.811785,...,-6.990570,-2.437605,-3.594774,1.345320,-4.257673,-2.776561,-0.907695,-1.865021,-0.675297,healthy
4,002d28bc-7806-4dfb-9c9b-afa8cb623cac.wav,0.466839,0.038284,2542.551657,1859.665993,4716.917928,0.234224,-440.167755,35.146564,-33.713917,...,-6.674835,3.067164,-1.296608,4.719309,-6.739134,-3.281385,-3.970852,-1.485659,-2.427423,healthy


## 失敗檔案檢查（選用）

In [8]:
if errors:
    print(pd.DataFrame(errors).to_string())
else:
    print('沒有失敗的檔案！')

沒有失敗的檔案！
